# AD gap pipeline walkthrough

The whole pipeline on one video, step by step: VAD and Whisper to find the
gaps, a corpus lookup and verbal-deixis cross-reference, PANNs audio
classification, Qwen3-VL clip descriptions, and a motion and cut signal.

Every step calls the code in `src/`; nothing is reimplemented here. The
notebook is generated by `notebooks/build_notebook.py`, so edit that and
regenerate rather than editing the `.ipynb`. The design rationale is in the
[README](../README.md).

## Stages

1. **VAD + Whisper.** Silero VAD finds speech; everything else is a gap.
   Whisper transcribes only the speech regions, so it can't hallucinate over
   silence. This step also looks up the video's corpus metadata and runs the
   verbal-deixis cross-reference.
2. **PANNs audio classification.** Each gap's audio goes into silence,
   music, sfx, ambient or speech.
3. **Qwen3-VL description.** Four structured fields (subject, action, scene,
   on-screen text) and a short free description, from the gap's video clip.
4. **Motion and cuts.** OpenCV frame differencing for motion and hard cuts,
   a model-free check on stage 3.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import json
import subprocess

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, Video, display

import gap_extract
import corpus_metadata
import deixis_crossref
import audio_classify
import frame_describe
import visual_signal

ROOT = Path.cwd().parent
RAW_VIDEOS = ROOT / "data" / "raw_videos"

WHISPER_MODEL = "mlx-community/whisper-large-v3-turbo"


In [ ]:
# Any YouTube Shorts ID works. It's downloaded below if it isn't already
# in data/raw_videos/.
VIDEO_ID = "IsEqr9BB2rc"

VIDEO_PATH = RAW_VIDEOS / f"{VIDEO_ID}.mp4"
if not VIDEO_PATH.exists():
    print(f"{VIDEO_PATH.name} not found locally, downloading...")
    subprocess.run(
        [
            "yt-dlp", "-f", "mp4/best", "-o", str(RAW_VIDEOS / f"{VIDEO_ID}.%(ext)s"),
            f"https://www.youtube.com/shorts/{VIDEO_ID}",
        ],
        check=True,
    )

duration = gap_extract.probe_duration(VIDEO_PATH)
print(f"{VIDEO_PATH.name}: {duration:.1f}s")


In [ ]:
Video(str(VIDEO_PATH), embed=True, width=280)


## Corpus metadata

Category and subgenre, plus the deixis labels if the video was hand-coded.
Blank for videos outside the corpus.

In [ ]:
meta = corpus_metadata.lookup(VIDEO_ID)
if not meta["in_corpus"]:
    print(f"{VIDEO_ID} is not in the corpus manifest (all_videos_master.csv).")
else:
    print(f"category={meta['category']}  subgenre={meta['subgenre']}  channel={meta['seed_channel']}")
    if meta["hand_coded"] == "Y":
        print(f"\nHand-coded: deixis_present={meta['deixis_present']}  "
              f"modality={meta['modality']}  levinson_type={meta['levinson_type']}")
        if meta["referent_note"]:
            print(f"referent_note: {meta['referent_note']!r}")
    else:
        print("\nNot in the 730-video hand-coded subset.")


## Step 1: audio waveform

In [ ]:
wav = gap_extract.load_wav_16k(VIDEO_PATH)
audio = wav.numpy()
t = np.linspace(0, duration, len(audio))

fig, ax = plt.subplots(figsize=(11, 2.5))
ax.plot(t, audio, linewidth=0.4, color="#444")
ax.set_xlim(0, duration)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude")
ax.set_title(f"{VIDEO_PATH.stem}: raw waveform")
fig.tight_layout()
plt.show()

# The embedded <video> above often won't play in VS Code's notebook viewer.
# Audio() uses a different renderer that does, and the 16 kHz audio is
# already loaded.
display(Audio(data=audio, rate=16000))


## Step 2: voice activity detection (Silero VAD)

VAD decides where the gaps are, not Whisper's segments, because Whisper
hallucinates over silence (see the `gap_extract.py` docstring).


In [ ]:
speech_regions = gap_extract.detect_speech_regions(VIDEO_PATH, duration)

vad_df = pd.DataFrame(speech_regions, columns=["start", "end"])
vad_df["duration"] = vad_df["end"] - vad_df["start"]
vad_df


In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.5))
ax.plot(t, audio, linewidth=0.4, color="#444")
for start, end in speech_regions:
    ax.axvspan(start, end, color="#4C9F70", alpha=0.3)
ax.set_xlim(0, duration)
ax.set_xlabel("time (s)")
ax.set_title(f"{VIDEO_PATH.stem}: waveform with VAD speech regions (green)")
fig.tight_layout()
plt.show()


## Step 3: transcribe the speech regions (Whisper)

In [ ]:
result = gap_extract.transcribe_regions(VIDEO_PATH, speech_regions, WHISPER_MODEL)
speech_spans = gap_extract.clean_speech_spans(result)

spans_df = pd.DataFrame([s.__dict__ for s in speech_spans])
spans_df


## Step 4: gaps are everything outside the speech regions

In [ ]:
gaps = gap_extract.compute_gaps(speech_regions, duration)

gaps_df = pd.DataFrame([g.__dict__ for g in gaps])
gaps_df


In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.5))
ax.plot(t, audio, linewidth=0.4, color="#444")
for start, end in speech_regions:
    ax.axvspan(start, end, color="#4C9F70", alpha=0.3, label="speech (VAD)")
for g in gaps:
    ax.axvspan(g.start, g.end, color="#C44E52", alpha=0.4, label="gap")
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc="upper right")
ax.set_xlim(0, duration)
ax.set_xlabel("time (s)")
ax.set_title(f"{VIDEO_PATH.stem}: speech (green) vs gaps (red)")
fig.tight_layout()
plt.show()

gap_ratio = sum(g.duration for g in gaps) / duration
print(f"gap-to-runtime ratio: {gap_ratio:.1%}  ({len(gaps)} gaps)")


## Step 5: verbal-deixis cross-reference

How close each deictic word ("this", "here", ...) is to a gap. Spoken deixis
only: gestural deixis has no timestamps in the hand-coded corpus. The
detector over-fires, so treat the counts as a screening signal.


In [ ]:
xref = deixis_crossref.compute_crossref(result, [(g.start, g.end) for g in gaps], duration)

print(f"{xref['n_verbal_deixis_events']} verbal deixis event(s) detected "
      f"({xref['verbal_deixis_density_per_min']}/min)")
for e in xref["events"]:
    print(f"  [{e['start']:.2f}-{e['end']:.2f}]  {e['text']!r}  supported={e['supported']}")

print()
coverage_df = pd.DataFrame(xref["coverage_by_threshold"])
coverage_df


## Step 6: the record `gap_extract.py` saves to `data/gaps/<video_id>.json`

In [ ]:
record = gap_extract.process_video(VIDEO_PATH, WHISPER_MODEL)
print(json.dumps(record, indent=2)[:2000], "\n...")


## Step 7: classify each gap's audio (PANNs)

Each gap goes into silence, music, sfx or ambient, using AudioSet's
ontology. Gaps shorter than Cnn14's minimum input (about 1 s) are padded
with neighbouring audio. The padding is reported, and a fifth bucket,
speech, shows when it pulled in narration.


In [ ]:
gap_classifications = []
for g in gaps:
    result_cls = audio_classify.classify_gap(VIDEO_PATH, g.start, g.end, duration)
    gap_classifications.append(result_cls)
    print(f"gap [{g.start:.2f}-{g.end:.2f}]  dur={g.duration:.2f}s  pos={g.position}")
    print(f"  classified over [{result_cls.classify_start:.2f}-{result_cls.classify_end:.2f}]"
          f"  (padded {result_cls.padded_seconds:.2f}s)")
    print(f"  -> bucket={result_cls.bucket}  top={result_cls.top_label} ({result_cls.top_prob:.2f})")
    for lbl, p in result_cls.top5:
        print(f"     {lbl}: {p:.3f}")
    print()


In [ ]:
n_speech_leak = sum(1 for c in gap_classifications if c.bucket == "speech")
n_padded = sum(1 for c in gap_classifications if c.padded_seconds > 0)
if n_speech_leak:
    print(f"{n_speech_leak}/{len(gaps)} gap(s) classified as 'speech': padding pulled in "
          f"adjacent narration. Expected for very short gaps.")
elif n_padded:
    print(f"{n_padded}/{len(gaps)} gap(s) needed padding but none leaked into 'speech'.")
else:
    print("All gaps were long enough to classify without padding.")


## Step 8: describe each gap's clip (Qwen3-VL)

Qwen3-VL gets the gap as a multi-frame clip and returns four fields, based
on what AD guidelines agree should be described: **subject** (worded like
the corpus's `referent_note` field), **action**, **scene** continuity and
on-screen **text**, plus a short free **description**. There's no minimum
gap length, since the model always gets a few frames. This step is
exploratory; the gap statistics don't depend on it.

### Choosing the model

`Qwen3-VL-8B-Instruct-4bit` described one hard clip, a 60 s fast-cutting
gap with no speech (the whole video is one gap), as "a static promotional
image... a central hourglass". The frames show a montage of sea creatures.
All 8 Qwen3-VL variants (2B, 4B, 8B and 30B-A3B, each as Instruct and
Thinking) were run on that clip, and the three strongest again on three
ordinary videos (a golden retriever, two bat clips) with hand-coded labels
to check against.

Every model handled the ordinary videos; the failures (hallucination,
reasoning loops, ignoring the output format) only showed up on the hard
clip. `Qwen3-VL-4B-Instruct-4bit` was the only model right on all four
videos, and also the smallest and fastest, so it's the default.
`8B-Instruct` hallucinates on hard content with no sign that it has.
`8B-Thinking` loops unless `repetition_penalty` is set, so it isn't safe to
run unattended; `describe_gap(..., repetition_penalty=1.3)` is there for
rerunning one gap whose description looks wrong. `30B-A3B` wasn't worth its
size. Four videos is a small test, and the ranking could change at scale.


In [ ]:
gap_descriptions = []
for g in gaps:
    desc = frame_describe.describe_gap(VIDEO_PATH, g.start, g.end)
    gap_descriptions.append(desc)
    print(f"gap [{g.start:.2f}-{g.end:.2f}]  ({desc.sampled_frame_count} frames sampled "
          f"by the model, out of {desc.clip_total_frames} in the clip)")
    print(f"  subject: {desc.subject}")
    print(f"  action:  {desc.action}")
    print(f"  scene:   {desc.scene}")
    print(f"  text:    {desc.on_screen_text}")
    print(f"  description: {desc.description}\n")

# Save this video's descriptions to data/descriptions/<video_id>.json.
saved_to = frame_describe.save_descriptions(VIDEO_ID, gaps, gap_descriptions)
print(f"saved -> {saved_to}")


In [ ]:
# up to 3 representative frames per gap
n_show = 3
fig, axes = plt.subplots(len(gap_descriptions), n_show, figsize=(3.5 * n_show, 3.2 * len(gap_descriptions)))
axes = np.atleast_2d(axes).reshape(len(gap_descriptions), n_show)

for row, desc in enumerate(gap_descriptions):
    cap = cv2.VideoCapture(str(desc.clip_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(total - 1, 0), n_show).round().astype(int)
    for col, idx in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        ax = axes[row, col]
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"gap [{desc.clip_start:.1f}-{desc.clip_end:.1f}]  frame {col+1}/{n_show}", fontsize=8)
        ax.axis("off")
    cap.release()

fig.tight_layout()
plt.show()


## Step 9: motion and cuts (a model-free check on Qwen3-VL's action and scene fields)

In [ ]:
gap_signals = []
for g in gaps:
    sig = visual_signal.analyze_gap(VIDEO_PATH, g.start, g.end)
    gap_signals.append(sig)
    print(f"gap [{g.start:.2f}-{g.end:.2f}]  motion={sig.motion_magnitude}  "
          f"has_cut={sig.has_cut}  ({sig.n_frames_analyzed} frames)")


## Step 10: everything together, per gap

In [ ]:
def nearest_deixis_distance(gap, events):
    if not events:
        return None
    dists = [max(0.0, e["start"] - gap.end, gap.start - e["end"]) for e in events]
    return round(min(dists), 2)

summary_rows = []
for g, cls, desc, sig in zip(gaps, gap_classifications, gap_descriptions, gap_signals):
    summary_rows.append({
        "start": round(g.start, 2),
        "end": round(g.end, 2),
        "duration": round(g.duration, 2),
        "position": g.position,
        "nearest_verbal_deixis_s": nearest_deixis_distance(g, xref["events"]),
        "audio_bucket": cls.bucket,
        "subject": desc.subject,
        "action": desc.action,
        "scene": desc.scene,
        "on_screen_text": desc.on_screen_text,
        "description": desc.description,
        "motion": sig.motion_magnitude,
        "has_cut": sig.has_cut,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df
